In [ ]:
'''
BT 2 - Data Collection.ipynb
Author: Muhammmad Talha & Jingchuan Shi
Supervisor: Dr. Ahmed Qureshi
Created 2019/9/6, last modified 2026/02/02 at University of Alberta.
All Rights Reserved.
'''

# === BT-2: Collection via spaCy + WordNet (replaces stanza/StanfordNLP) ===
from pathlib import Path
import spacy
from nltk.corpus import wordnet as wn

try:
    nlp = spacy.load("en_core_web_lg")
except OSError:
    raise RuntimeError("Install model via: python -m spacy download en_core_web_lg")

nlp.max_length = 3_000_000

def is_valid_verb(w: str) -> bool:
    return any(ss.pos() == "v" for ss in wn.synsets(w))

def extract_verbs(text: str):
    doc = nlp(text)
    out=[]
    for tok in doc:
        if tok.pos_ in {"VERB","AUX"}:
            lem = tok.lemma_.lower().strip()
            if lem.isalpha() and is_valid_verb(lem):
                out.append(lem)
    return out


In [ ]:
# ===================== BT-2: Collect new verbs =====================
from pathlib import Path
import stanza
from nltk.corpus import wordnet as wn

# 0) NLP pipeline (reuse resources; same setup you’ve been using)
try:
    from stanza.pipeline.core import DownloadMethod
    nlp = stanza.Pipeline(
        lang="en",
        processors="tokenize,pos,lemma",
        tokenize_pretokenized=False,
        download_method=DownloadMethod.REUSE_RESOURCES
    )
except Exception:
    nlp = stanza.Pipeline(lang="en", processors="tokenize,pos,lemma", tokenize_pretokenized=False)

# 1) Root + standard folders (unchanged setup)
BASE_DIR   = Path.cwd()
RESULTS_DIR = BASE_DIR / "results"
LOGS_DIR    = BASE_DIR / "logs"
CORPUS_DIR  = BASE_DIR / "data"
for d in (RESULTS_DIR, LOGS_DIR, CORPUS_DIR):
    d.mkdir(parents=True, exist_ok=True)

core_source_path = RESULTS_DIR / "BTverblist_core.txt"    # from BT-1
new_source_path  = RESULTS_DIR / "BTverblist_new.txt"     # grows here
done_source_path = RESULTS_DIR / "BTdone_paper_list.txt"  # processed files
progress_path    = LOGS_DIR    / "BTgain_progress.txt"    # progress log
for p in (core_source_path, new_source_path, done_source_path, progress_path):
    p.touch(exist_ok=True)

# 2) Helpers (same robust IO)
def extract_text(file_path: Path) -> str:
    """Read text from .pdf (via Tika) or .txt (utf-8). Return '' if not available."""
    if file_path.suffix.lower() == ".pdf":
        try:
            from tika import parser as tika_parser
            raw = tika_parser.from_file(str(file_path))
            txt = (raw or {}).get("content") or ""
            return txt
        except Exception:
            return ""
    else:  # .txt
        try:
            return file_path.read_text(encoding="utf-8", errors="ignore")
        except Exception:
            return ""

def as_abs_str(p: Path) -> str:
    try:
        return str(p.resolve())
    except Exception:
        return str(p.absolute())

# 3) Load existing lists (unchanged)
core_verbs = [l.strip() for l in core_source_path.read_text(encoding="utf-8").splitlines() if l.strip()]
verbs      = [l.strip() for l in new_source_path.read_text(encoding="utf-8").splitlines() if l.strip()]
old_size   = len(verbs)
print(f"\nCurrent data size: {old_size}")

done_papers = {line.strip() for line in done_source_path.read_text(encoding="utf-8").splitlines() if line.strip()}
done_abs = {as_abs_str(Path(s)) for s in done_papers}

# 4) Files to process (same recursive scan you’re using)
files = []
for p in CORPUS_DIR.rglob("*"):
    if p.is_file() and p.suffix.lower() in {".pdf", ".txt"} and as_abs_str(p) not in done_abs:
        files.append(p)

core_set  = set(core_verbs)
verbs_set = set(verbs)

# 5) LEGACY VERB CRITERION (only change you asked for)
#    - Use XPOS only: {VB, VBD, VBG, VBN, VBP, VBZ}
#    - Lemma lowercased; no extra normalization/regex
VB_TAGS = {"VB","VBD","VBG","VBN","VBP","VBZ"}

for file_path in files:
    raw_text = extract_text(file_path)
    if not raw_text:
        with done_source_path.open("a", encoding="utf-8", newline="\n") as n:
            n.write(as_abs_str(file_path) + "\n")
        done_abs.add(as_abs_str(file_path))
        continue

    # basic PDF hyphenation/newline cleanup
    raw_text = raw_text.replace("-\n", "").replace("\n", " ")

    # NLP pass
    try:
        doc = nlp(raw_text)
    except Exception:
        with done_source_path.open("a", encoding="utf-8", newline="\n") as n:
            n.write(as_abs_str(file_path) + "\n")
        done_abs.add(as_abs_str(file_path))
        continue

    new_verbs = []
    for sent in doc.sentences:
        for w in sent.words:
            if w.xpos in VB_TAGS:
                lemma = (w.lemma or "").lower()
                if lemma and lemma not in core_set and lemma not in verbs_set and lemma not in new_verbs:
                    new_verbs.append(lemma)
                    verbs.append(lemma)
                    verbs_set.add(lemma)

    # Append new verbs (unchanged IO)
    if new_verbs:
        with new_source_path.open("a", encoding="utf-8", newline="\n") as m:
            for w in new_verbs:
                m.write(w + "\n")

    # Mark file as processed (unchanged absolute-path tracking)
    with done_source_path.open("a", encoding="utf-8", newline="\n") as n:
        n.write(as_abs_str(file_path) + "\n")
    done_abs.add(as_abs_str(file_path))

    # Progress report & log (unchanged)
    gain = len(verbs) - old_size
    print(f"Loaded {file_path.name} — total verbs: {len(verbs)} (+{gain}).")
    old_size = len(verbs)
    with progress_path.open("a", encoding="utf-8", newline="\n") as n:
        n.write(f"{len(verbs)} {gain}\n")

# 6) WordNet validation (unchanged)
valid_verbs = [v for v in verbs if wn.synsets(v, pos="v")]
print("# of verbs identified by WordNet:", len(valid_verbs))


2025-10-01 15:47:06 WARNING: Language en package default expects mwt, which has been added
2025-10-01 15:47:06 INFO: Loading these models for language: en (English):
| Processor | Package           |
---------------------------------
| tokenize  | combined          |
| mwt       | combined          |
| pos       | combined_charlm   |
| lemma     | combined_nocharlm |

2025-10-01 15:47:06 INFO: Using device: cpu
2025-10-01 15:47:06 INFO: Loading: tokenize
2025-10-01 15:47:08 INFO: Loading: mwt
2025-10-01 15:47:08 INFO: Loading: pos
2025-10-01 15:47:10 INFO: Loading: lemma
2025-10-01 15:47:10 INFO: Done loading processors!



Current data size: 4391
# of verbs identified by WordNet: 2464


In [ ]:
# ===== BT-2: Rebuild list from corpus (force reprocess, legacy criterion) =====
from pathlib import Path
import stanza
from nltk.corpus import wordnet as wn

# 0) Setup (unchanged dirs)
BASE_DIR   = Path.cwd()
RESULTS_DIR = BASE_DIR / "results"
LOGS_DIR    = BASE_DIR / "logs"
CORPUS_DIR  = BASE_DIR / "data"
for d in (RESULTS_DIR, LOGS_DIR, CORPUS_DIR):
    d.mkdir(parents=True, exist_ok=True)

core_source_path = RESULTS_DIR / "BTverblist_core.txt"
new_source_path  = RESULTS_DIR / "BTverblist_new.txt"
done_source_path = RESULTS_DIR / "BTdone_paper_list.txt"
progress_path    = LOGS_DIR    / "BTgain_progress.txt"
for p in (core_source_path, new_source_path, done_source_path, progress_path):
    p.touch(exist_ok=True)

# 1) NLP
try:
    from stanza.pipeline.core import DownloadMethod
    nlp = stanza.Pipeline(lang="en", processors="tokenize,pos,lemma",
                          tokenize_pretokenized=False,
                          download_method=DownloadMethod.REUSE_RESOURCES)
except Exception:
    nlp = stanza.Pipeline(lang="en", processors="tokenize,pos,lemma", tokenize_pretokenized=False)

# 2) Load lists
core_verbs = [l.strip() for l in core_source_path.read_text(encoding="utf-8").splitlines() if l.strip()]
verbs      = [l.strip() for l in new_source_path.read_text(encoding="utf-8").splitlines() if l.strip()]
old_size   = len(verbs)
print(f"\nCurrent data size: {old_size}")

# 3) Build file list (IGNORE done list for this run)
files = [p for p in CORPUS_DIR.iterdir()
         if p.is_file() and p.suffix.lower() in {".pdf", ".txt"}]

# 4) Readers
def extract_text(p: Path) -> str:
    if p.suffix.lower() == ".pdf":
        try:
            from tika import parser as tika_parser
            raw = tika_parser.from_file(str(p))
            return (raw or {}).get("content") or ""
        except Exception:
            return ""
    try:
        return p.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        return ""

# 5) Legacy verb criterion (XPOS VB* only)
VB_TAGS = {"VB","VBD","VBG","VBN","VBP","VBZ"}
core_set  = set(core_verbs)
verbs_set = set(verbs)

for fp in files:
    txt = extract_text(fp)
    if not txt:
        print(f"(skip unreadable) {fp.name}")
        continue
    txt = txt.replace("-\n","").replace("\n"," ")

    try:
        doc = nlp(txt)
    except Exception:
        print(f"(skip nlp error) {fp.name}")
        continue

    new_verbs = []
    for sent in doc.sentences:
        for w in sent.words:
            if w.xpos in VB_TAGS:
                lemma = (w.lemma or "").lower()
                if lemma and lemma not in core_set and lemma not in verbs_set and lemma not in new_verbs:
                    new_verbs.append(lemma)
                    verbs.append(lemma)
                    verbs_set.add(lemma)

    if new_verbs:
        with new_source_path.open("a", encoding="utf-8", newline="\n") as m:
            for w in new_verbs:
                m.write(w + "\n")

    # (We do NOT touch the done list here; we’re force-reprocessing if run again)

    gain = len(verbs) - old_size
    print(f"Processed {fp.name} — total verbs: {len(verbs)} (+{gain}).")
    old_size = len(verbs)
    with progress_path.open("a", encoding="utf-8", newline="\n") as n:
        n.write(f"{len(verbs)} {gain}\n")

# 6) WordNet validation
wn_verbs = [v for v in verbs if wn.synsets(v, pos="v")]
print("# of verbs identified by WordNet:", len(wn_verbs))



2025-10-01 15:50:28 WARNING: Language en package default expects mwt, which has been added
2025-10-01 15:50:28 INFO: Loading these models for language: en (English):
| Processor | Package           |
---------------------------------
| tokenize  | combined          |
| mwt       | combined          |
| pos       | combined_charlm   |
| lemma     | combined_nocharlm |

2025-10-01 15:50:28 INFO: Using device: cpu
2025-10-01 15:50:28 INFO: Loading: tokenize
2025-10-01 15:50:28 INFO: Loading: mwt
2025-10-01 15:50:28 INFO: Loading: pos
2025-10-01 15:50:30 INFO: Loading: lemma
2025-10-01 15:50:30 INFO: Done loading processors!



Current data size: 4391


2025-10-01 15:50:32,803 [MainThread  ] [INFO ]  Retrieving http://search.maven.org/remotecontent?filepath=org/apache/tika/tika-server-standard/3.1.0/tika-server-standard-3.1.0.jar to C:\Users\talha4\AppData\Local\Temp\tika-server.jar.
2025-10-01 15:50:34,854 [MainThread  ] [INFO ]  Retrieving http://search.maven.org/remotecontent?filepath=org/apache/tika/tika-server-standard/3.1.0/tika-server-standard-3.1.0.jar.md5 to C:\Users\talha4\AppData\Local\Temp\tika-server.jar.md5.
2025-10-01 15:50:35,960 [MainThread  ] [WARNI]  Failed to see startup log message; retrying...


Processed 0211.pdf — total verbs: 4391 (+0).
Processed 064325b03389e181f226cea9c502ffa0bf19.pdf — total verbs: 4391 (+0).


KeyboardInterrupt: 

In [6]:
import pathlib, os
print("BT‑n notebook cwd:", pathlib.Path.cwd())
print("core file exists :", pathlib.Path("results/BTverblist_core.txt").exists())


BT‑n notebook cwd: C:\Users\talha4\Downloads\Bloom-s-Taxonomy-extension-master\Bloom-s-Taxonomy-extension-master
core file exists : True
